# Skin Lesion Segmentation Experiments

This Google-Colab-compatible notebook compares Otsu, K-means + morphology, and a compact U-Net on the small official ISIC Task 1 bundle. The default is ISIC 2016 (900 image/mask pairs; about 602 MB + 6 MB); the much larger ISIC 2018 archive remains optional through the Kaggle/API loaders. It uses real downloaded image/mask pairs when the experiment cells are run. The project is educational and not diagnostic.

## 1. Environment setup

In [ ]:
%pip install -q opencv-python-headless numpy pandas matplotlib scikit-image scikit-learn Pillow requests tensorflow
import os, random, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import tensorflow as tf
sys.path.insert(0, str(Path.cwd()))
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print({"python": sys.version, "numpy": np.__version__, "opencv": cv2.__version__, "pandas": pd.__version__, "tensorflow": tf.__version__})

## 2. Configuration and project imports

Set `NUM_SAMPLES` to 20 for debugging, 100 for quick validation, 500 for development, or `'all'` for the main 900-pair run. The U-Net section uses explicit train/validation/test splits and does not score training images.

In [ ]:
DATA_SOURCE = 'official_isic'
NUM_SAMPLES = 'all'
TARGET_SIZE = (256, 256)
PROJECT_ROOT_OVERRIDE = os.environ.get('SKIN_LESION_PROJECT_ROOT')
PROJECT_CANDIDATES = []
if PROJECT_ROOT_OVERRIDE:
    PROJECT_CANDIDATES.append(Path(PROJECT_ROOT_OVERRIDE))
PROJECT_CANDIDATES.extend([Path.cwd(), Path.home() / 'MYfile' / 'Desktop' / 'PYThh' / 'Skin - Les Analyzer', Path('/content/skin-lesion-analyzer')])
PROJECT_ROOT = next((p for p in PROJECT_CANDIDATES if (p / 'analyzer' / 'data_loader.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Local project code was not found. Start Jupyter from the repository root or set SKIN_LESION_PROJECT_ROOT to the repository path, then rerun this cell.')
sys.path.insert(0, str(PROJECT_ROOT))
if 'analyzer' in sys.modules: del sys.modules['analyzer']
CACHE_DIR = PROJECT_ROOT / 'dataset_cache'
print('Using project:', PROJECT_ROOT)
from analyzer.data_loader import deterministic_split, download_dataset, pair_image_masks
from analyzer.preprocessing import resize_image
from analyzer.classical_segmentation import otsu_segmentation, kmeans_segmentation
from analyzer.metrics import dice_score, iou_score, summarize_scores
from analyzer.unet_segmentation import build_unet, compile_unet, dice_bce_loss, dice_metric


## 3. Download and verify official ISIC pairs

The default official route downloads the small ISIC Task 1 training archives and caches them outside Git. Pairing validation raises an error for any unmatched stem rather than silently dropping data. KaggleHub remains an optional source in the loader.

In [ ]:
image_dir, mask_dir, dataset_metadata = download_dataset(source=DATA_SOURCE, cache_dir=CACHE_DIR, split='training')
pairs = pair_image_masks(image_dir, mask_dir)
if str(NUM_SAMPLES).lower() != 'all': pairs = pairs[:int(NUM_SAMPLES)]
train_pairs, val_pairs, test_pairs = deterministic_split(pairs, seed=SEED)
print('verified pairs:', len(pairs), '| train:', len(train_pairs), '| validation:', len(val_pairs), '| test:', len(test_pairs))
assert len(pairs) >= 3

In [ ]:
def load_pair(pair):
    image = np.asarray(Image.open(pair[0]).convert('RGB'))
    mask = np.asarray(Image.open(pair[1]).convert('L'))
    image = resize_image(image, TARGET_SIZE)
    mask = cv2.resize(mask, (TARGET_SIZE[1], TARGET_SIZE[0]), interpolation=cv2.INTER_NEAREST) > 0
    return image, mask.astype(np.uint8)
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, pair in zip(axes.flat, pairs[:6]):
    image, mask = load_pair(pair); ax.imshow(image); ax.contour(mask, levels=[0.5], colors='lime'); ax.set_title(pair[0].stem); ax.axis('off')
plt.tight_layout()

## 4. Classical baselines

Both methods are scored on the same verified samples. No metric value is inserted manually.

In [ ]:
def evaluate_classical(method):
    ious, dices = [], []
    for pair in test_pairs:
        image, truth = load_pair(pair)
        pred = otsu_segmentation(image) if method == 'Otsu' else kmeans_segmentation(image)
        ious.append(iou_score(pred, truth)); dices.append(dice_score(pred, truth))
    return summarize_scores(ious, dices)
classical_results = {m: evaluate_classical(m) for m in ['Otsu', 'K-means + Morphology']}
classical_results

## 5. U-Net train/validation/test split

This is the offline training section. The best checkpoint is selected on validation loss, and the final U-Net metrics are calculated only on the held-out test split. For a complete manifest, per-sample CSV, JSON metadata, and presentation figures, the equivalent `scripts/train_unet.py` and `scripts/run_benchmark.py` commands are authoritative.

In [ ]:
from sklearn.model_selection import train_test_split
trainval_pairs = train_pairs + val_pairs
loaded = [load_pair(pair) for pair in trainval_pairs]
images = np.stack([x for x, _ in loaded]).astype('float32') / 255.0
masks = np.stack([y for _, y in loaded]).astype('float32')[..., None]
indices = np.arange(len(trainval_pairs))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=SEED)
model = compile_unet(build_unet((*TARGET_SIZE, 3)))
model_path = PROJECT_ROOT / 'model' / 'unet_skin_lesion.keras'
callbacks = [tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(patience=2), tf.keras.callbacks.ModelCheckpoint(model_path, save_best_only=True)]
history = model.fit(images[train_idx], masks[train_idx], validation_data=(images[val_idx], masks[val_idx]), epochs=25, batch_size=4, callbacks=callbacks)

In [ ]:
unet_ious, unet_dices = [], []
test_loaded = [load_pair(pair) for pair in test_pairs]
test_images = np.stack([x for x, _ in test_loaded]).astype('float32') / 255.0
test_masks = np.stack([y for _, y in test_loaded]).astype('uint8')
for idx in range(len(test_images)):
    pred = (model.predict(test_images[idx:idx+1], verbose=0)[0, ..., 0] >= 0.5).astype(np.uint8)
    truth = test_masks[idx]
    unet_ious.append(iou_score(pred, truth)); unet_dices.append(dice_score(pred, truth))
unet_results = summarize_scores(unet_ious, unet_dices)
unet_results

## 6. Aggregate results, select the measured best method, and visualize

The selection is by mean Dice, breaking ties with mean IoU. It is not assumed that U-Net wins.

In [ ]:
rows = [{'dataset': dataset_metadata.get('dataset', 'ISIC2018_Task1'), 'source': dataset_metadata['source'], 'split': dataset_metadata['split'], 'evaluation_split': 'test', 'method': name, **scores} for name, scores in classical_results.items()]
rows.append({'dataset': dataset_metadata.get('dataset', 'ISIC2018_Task1'), 'source': dataset_metadata['source'], 'split': dataset_metadata['split'], 'evaluation_split': 'test', 'method': 'U-Net', **unet_results})
results = pd.DataFrame(rows).sort_values(['mean_dice', 'mean_iou'], ascending=False)
results.to_csv(PROJECT_ROOT / 'results/benchmark_results.csv', index=False)
print(results.to_string(index=False))
print('selected method:', results.iloc[0]['method'])

In [ ]:
best = results.iloc[0]['method']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
image, truth = load_pair(test_pairs[0])
pred_otsu = otsu_segmentation(image); pred_kmeans = kmeans_segmentation(image)
pred_unet = (model.predict(image[None].astype('float32') / 255.0, verbose=0)[0, ..., 0] >= 0.5).astype(np.uint8)
for ax, title, view in zip(axes, ['Original', 'Ground Truth', 'Otsu', 'K-means', 'U-Net'], [image, truth, pred_otsu, pred_kmeans, pred_unet]): ax.imshow(view, cmap=None if view.ndim == 3 else 'gray'); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.savefig(PROJECT_ROOT / 'results/figures/comparison.png', dpi=150)